In [ ]:
#Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cross_decomposition import PLSRegression
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5
N_PLS_COMPONENTS = 10

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "05_text_speech_eeg.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"


In [ ]:
class PLSDA(BaseEstimator, TransformerMixin):
    # PLS-DA 

    def __init__(self, n_components=10):
        self.n_components = n_components

    def fit(self, X, y):
        n_comp = min(self.n_components, X.shape[1], X.shape[0] - 1)
        if n_comp < 1:
            raise ValueError("No hay suficientes muestras/variables para calcular PLS.")
        self.n_components_ = n_comp
        self.pls_ = PLSRegression(n_components=n_comp, scale=False)
        self.pls_.fit(X, y)
        return self

    def transform(self, X):
        return self.pls_.transform(X)


def load_partitions():
    """Carga directamente las particiones del paper."""
    partitions = pd.read_csv(PARTITIONS_PATH)
    return partitions[["subject_id", "avatar", "outer_fold"]].copy()


def get_metrics(y_true, y_pred, y_prob):
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def summarize_metrics(metrics_df, metric_cols):
    return pd.concat([
        metrics_df[metric_cols].mean().round(3).rename("mean"),
        metrics_df[metric_cols].std().round(3).rename("std"),
    ], axis=1)


def subject_predictions(pred_conv):
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["prob_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["prob_1"] >= 0.5).astype(int)
    return pred_subject


def build_model(text_cols, speech_cols, eeg_cols):
    text_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pls", PLSDA(n_components=N_PLS_COMPONENTS)),
    ])

    speech_pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pls", PLSDA(n_components=N_PLS_COMPONENTS)),
    ])

    # imputación + escalado dentro del pipeline.
    eeg_pipe = Pipeline([
        ("scaler", StandardScaler()),
    ])

    preprocessor = ColumnTransformer([
        ("text_pls", text_pipe, text_cols),
        ("speech_pls", speech_pipe, speech_cols),
        ("eeg", eeg_pipe, eeg_cols),
    ])

    model = Pipeline([
        ("prep", preprocessor),
        ("xgb", XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=-1)),
    ])
    return model


def get_param_grid():
    return {
        "xgb__max_depth": list(range(3, 12)),
        "xgb__n_estimators": [25, 50, 100, 200],
    }


def get_scoring():
    return {
        "WAcc": "accuracy",
        "UAcc": "balanced_accuracy",
        "auc": "roc_auc",
        "f1": "f1",
        "precision": "precision",
        "recall": "recall",
    }


In [ ]:
data = pd.read_csv(INPUT_PATH)
partitions = load_partitions()

text_cols = sorted([c for c in data.columns if c.startswith("text_")], key=lambda c: int(c.split("_", 1)[1]))
speech_cols = sorted([c for c in data.columns if c.startswith("speech_")], key=lambda c: int(c.split("_", 1)[1]))

meta_cols = {
    "subject_id", "avatar", "label", "outer_fold", "emotion", "narrative",
    "conversation_folder", "conversation_id", "conversation_time", "phq",
    "time_diff_seconds", "text_path", "speech_path", "eeg_path", "has_eeg",
}

eeg_cols = [
    c for c in data.columns
    if c not in meta_cols and c not in text_cols and c not in speech_cols
    and pd.api.types.is_numeric_dtype(data[c])
]

partitions = shuffle(partitions, random_state=SEED).reset_index(drop=True)

df = data.merge(
    partitions[["subject_id", "avatar", "outer_fold"]],
    on=["subject_id", "avatar"],
    how="inner",
)

initial_rows = len(df)

# conserva solo conversaciones con text + speech + EEG.
feature_cols = text_cols + speech_cols + eeg_cols
df = df.dropna(subset=feature_cols).copy()

print("Filas iniciales con partición:", initial_rows)
print("Filas eliminadas por faltar alguna modalidad:", initial_rows - len(df))
print("Filas trimodales finales:", len(df))
print("Sujetos finales:", df["subject_id"].nunique())
print("Variables text originales:", len(text_cols))
print("Variables speech originales:", len(speech_cols))
print("Variables EEG:", len(eeg_cols))
print("Variables finales tras PLS + EEG:", 10 + 10 + len(eeg_cols))

display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))
display(pd.crosstab(df.drop_duplicates("subject_id")["outer_fold"], df.drop_duplicates("subject_id")["label"]))


Filas iniciales con partición: 600
Filas eliminadas por faltar alguna modalidad: 42
Filas trimodales finales: 558
Sujetos finales: 94
Variables text originales: 768
Variables speech originales: 1024
Variables EEG: 27
Variables finales tras PLS + EEG: 47


,n_subjects
outer_fold,
1,20
2,18
3,18
4,19
5,19


label,0,1
outer_fold,,
1,11,9
2,10,8
3,11,7
4,12,7
5,11,8


In [ ]:
OUT_DIR = DATA_DIR / "07_results_text_speech_eeg_emotion_wise_PLS10"
OUT_DIR.mkdir(parents=True, exist_ok=True)

metric_cols = ["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]
narratives = sorted(df["avatar"].unique())

all_predictions = []
rows_by_narrative = []
best_params_rows = []

for narrative in narratives:
    print(f"\n================ {narrative} ================")
    df_n = df[df["avatar"] == narrative].reset_index(drop=True)

    for fold in sorted(df_n["outer_fold"].unique()):
        dev = df_n[df_n["outer_fold"] != fold].reset_index(drop=True)
        test = df_n[df_n["outer_fold"] == fold].reset_index(drop=True)

        if test.empty or dev["label"].nunique() < 2:
            continue

        X_dev = dev[feature_cols]
        y_dev = dev["label"].astype(int)
        g_dev = dev["subject_id"]

        X_test = test[feature_cols]
        y_test = test["label"].astype(int)

        inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

        grid = GridSearchCV(
            estimator=build_model(text_cols, speech_cols, eeg_cols),
            param_grid=get_param_grid(),
            scoring=get_scoring(),
            refit="UAcc",
            cv=inner_cv,
            n_jobs=-1,
            verbose=0,
        )
        grid.fit(X_dev, y_dev, groups=g_dev)

        model = build_model(text_cols, speech_cols, eeg_cols)
        model.set_params(**grid.best_params_)
        model.fit(X_dev, y_dev)

        prob_1 = model.predict_proba(X_test)[:, 1]
        pred = (prob_1 >= 0.5).astype(int)

        pred_fold = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
        pred_fold["prob_1"] = prob_1
        pred_fold["pred"] = pred
        pred_fold["narrative_model"] = narrative
        all_predictions.append(pred_fold)

        metrics = get_metrics(y_test, pred, prob_1)
        metrics["narrative"] = narrative
        metrics["outer_fold"] = fold
        metrics["CV_f1"] = grid.cv_results_["mean_test_f1"][grid.best_index_]
        rows_by_narrative.append(metrics)

        best_params_rows.append({
            "narrative": narrative,
            "outer_fold": fold,
            "best_max_depth": grid.best_params_["xgb__max_depth"],
            "best_n_estimators": grid.best_params_["xgb__n_estimators"],
            "best_inner_UAcc": grid.cv_results_["mean_test_UAcc"][grid.best_index_],
            "best_inner_f1": grid.cv_results_["mean_test_f1"][grid.best_index_],
            "n_dev_subjects": dev["subject_id"].nunique(),
            "n_test_subjects": test["subject_id"].nunique(),
        })

    print("Modelos entrenados para", narrative)

predictions_df = pd.concat(all_predictions, ignore_index=True)
metrics_by_narrative_fold = pd.DataFrame(rows_by_narrative)
best_params_df = pd.DataFrame(best_params_rows)


subject_fold_predictions = (
    predictions_df
    .groupby(["subject_id", "label", "outer_fold"], as_index=False)
    .agg(prob_1=("prob_1", "mean"), n_predictions=("prob_1", "size"))
)
subject_fold_predictions["pred"] = (subject_fold_predictions["prob_1"] >= 0.5).astype(int)

subject_metrics_rows = []
for fold, fold_df in subject_fold_predictions.groupby("outer_fold"):
    metrics = get_metrics(fold_df["label"], fold_df["pred"], fold_df["prob_1"])
    metrics["outer_fold"] = fold
    subject_metrics_rows.append(metrics)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)

summary_subject = pd.DataFrame({
    "metric": ["Subject-level Test F1"],
    "mean": [subject_metrics_df["f1"].mean()],
    "std": [subject_metrics_df["f1"].std()],
}).round(3)

print("\nResultados por narrativa: Conversation-level Test F1")
display(
    metrics_by_narrative_fold
    .groupby("narrative")["f1"]
    .agg(["mean", "std"])
    .round(3)
)

print("\nResultado final agregado por sujeto")
display(summary_subject)

print("\nSubject-level Test metrics")
display(summarize_metrics(subject_metrics_df, metric_cols))

print("\nNúmero de predicciones usadas por sujeto")
display(subject_fold_predictions["n_predictions"].value_counts().sort_index().to_frame("n_subjects"))

metrics_by_narrative_fold.to_csv(OUT_DIR / "narrative_conversation_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
summary_subject.to_csv(OUT_DIR / "summary_subject_level.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_narrative_and_fold.csv", index=False)
predictions_df.to_csv(OUT_DIR / "conversation_predictions_by_narrative.csv", index=False)
subject_fold_predictions.to_csv(OUT_DIR / "subject_predictions_aggregated.csv", index=False)

print("\nArchivos guardados en:", OUT_DIR)



================ Angry ================
Modelos entrenados para Angry

================ Happy ================
Modelos entrenados para Happy

================ Neutral1 ================
Modelos entrenados para Neutral1

================ Neutral2 ================
Modelos entrenados para Neutral2

================ Relax ================
Modelos entrenados para Relax

================ Sad ================
Modelos entrenados para Sad

Resultados por narrativa: Conversation-level Test F1


,mean,std
narrative,,
Angry,0.430,0.121
Happy,0.511,0.151
Neutral1,0.230,0.118
Neutral2,0.430,0.283
Relax,0.337,0.239
Sad,0.473,0.127



Resultado final agregado por sujeto


,metric,mean,std
0,Subject-level Test F1,0.313,0.199



Subject-level Test metrics


,mean,std
WAcc,0.585,0.101
UAcc,0.541,0.107
auc,0.610,0.190
f1,0.313,0.199
precision,0.557,0.441
recall,0.259,0.204
kappa,0.085,0.225



Número de predicciones usadas por sujeto


,n_subjects
n_predictions,
3,2
6,92



Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/results_text_speech_eeg_emotion_wise_PLS10
